# Serbian Wikipedia → Verb Prefix Dataset (HF + Stanza)

This notebook builds a dataset like `serbian_verb_final_02_02_with_ruscorpora_500.csv` using the Hugging Face dataset `datatab/SrpWikiDataset` and Stanza for lemmatization.

## 1) Imports and Parameters

In [1]:
import os
import csv
import time
from collections import Counter

import stanza
from datasets import load_dataset
from huggingface_hub.utils import HfHubHTTPError

# Parameters
DATASET_NAME = "datatab/SrpWikiDataset"
SPLIT = "train"
MAX_CHARS = 50_000     # chunk size for Stanza
MAX_DOCS = 1000        # set to an int for quick tests
START_DOC = 1000       # skip first 1000 docs (process 1001-2000)
MIN_FREQ = 5         # filter threshold for final dataset
OUT_DIR = os.path.join("data", "csv")

SERBIAN_PREFIXES = [
    "do", "na", "o", "ob", "od", "po", "pre", "pri", "pro", "raz",
    "s", "sa", "su", "u", "uz", "v", "iz", "za", "nad", "pod",
    "pred", "bez", "ras", "pr", "ot", "prema", "protiv", "obez",
]

os.makedirs(OUT_DIR, exist_ok=True)

C:\Users\Анастасия Андреевна\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Load Stanza pipeline

In [2]:
# Download models if missing
try:
    stanza.download("sr", verbose=False)
except Exception:
    pass

nlp = stanza.Pipeline(
    "sr",
    processors="tokenize,pos,lemma",
    use_gpu=False,
    verbose=False,
)

## 3) Stream the dataset and collect verb lemma frequencies

## 2.5) Preview dataset (optional)


In [3]:
# Peek at a few examples from the dataset
token = os.getenv("HF_TOKEN")
ds_preview = load_dataset(DATASET_NAME, split=SPLIT, streaming=True, token=token)

print(f"Dataset: {DATASET_NAME}\n")
for i, example in enumerate(ds_preview):
    if i >= 3:  # Show first 3 examples
        break
    print(f"{'='*80}")
    print(f"Example {i+1}:")
    print(f"{'='*80}")
    text = example["text"]
    print(text[:600] if len(text) > 600 else text)  # First 600 chars
    print(f"\n[Total length: {len(text)} characters]\n")


Dataset: datatab/SrpWikiDataset

Example 1:
UTF-8 UTF-8 varijanta je najzgodnija za kodiranje većinski latiničnog teksta.

[Total length: 77 characters]

Example 2:
Dato je i kratko uputstvo za korišćenje te varijante u Microsoft Word-u, Netscape Composer-u i tekstualnom editoru Kate.

[Total length: 120 characters]

Example 3:
U tekstu su takođe preporučeni standardni Unicode fontovi koji omogućavaju laku prenosivost teksta sa računara na računar ili za objavljivanje teksta na Internet.

[Total length: 162 characters]



In [10]:
def iter_text_stream(dataset_name, split):
    token = os.getenv("HF_TOKEN")
    for attempt in range(5):
        try:
            ds = load_dataset(dataset_name, split=split, streaming=True, token=token)
            break
        except HfHubHTTPError as e:
            if e.response.status_code == 429:
                wait_time = 2 ** attempt
                print(f"Rate limited, waiting {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                raise
    else:
        raise RuntimeError("Failed to load dataset after retries")
    for row in ds:
        text = row.get("text")
        if text:
            yield text


def extract_verb_lemmas(text_iter, nlp, max_chars=50_000, max_docs=None, start_doc=0):
    lemma_counter = Counter()
    seen_docs = 0
    processed_docs = 0
    buf = []
    buf_len = 0

    def flush_buffer():
        nonlocal buf, buf_len, seen_docs, processed_docs
        if not buf:
            return
        doc_text = " ".join(buf)
        buf = []
        buf_len = 0
        seen_docs += 1
        if seen_docs <= start_doc:
            return
        if max_docs is not None and processed_docs >= max_docs:
            return
        processed_docs += 1
        doc = nlp(doc_text)
        for sent in doc.sentences:
            for word in sent.words:
                if word.upos in ("VERB", "AUX"):
                    lemma = (word.lemma or "").lower()
                    if lemma:
                        lemma_counter[lemma] += 1

    for text in text_iter:
        if max_docs is not None and processed_docs >= max_docs:
            break
        text = text.strip()
        if not text:
            continue
        buf.append(text)
        buf_len += len(text)
        if buf_len >= max_chars:
            flush_buffer()

    flush_buffer()
    return lemma_counter


text_iter = iter_text_stream(DATASET_NAME, SPLIT)
lemma_counter = extract_verb_lemmas(text_iter, nlp, max_chars=MAX_CHARS, max_docs=MAX_DOCS, start_doc=START_DOC)

len(lemma_counter)

KeyboardInterrupt: 

## 4) Save lemma frequencies

In [4]:
lemma_csv = os.path.join(OUT_DIR, "serbian_verbs_frequency_hf_7001_10000.csv")
rows = sorted(lemma_counter.items(), key=lambda x: x[1], reverse=True)
with open(lemma_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["глагол", "частота"])
    writer.writerows(rows)

lemma_csv

'data\\csv\\serbian_verbs_frequency_hf_7001_10000.csv'

## 5) Build prefix pairs

In [5]:
freq_dict = dict(lemma_counter)
pairs = []

for prefixed in freq_dict.keys():
    for prefix in SERBIAN_PREFIXES:
        if prefixed.startswith(prefix) and len(prefixed) > len(prefix) + 2:
            base = prefixed[len(prefix):]
            if base in freq_dict:
                pairs.append(
                    (
                        base,
                        prefixed,
                        prefix,
                        freq_dict[base],
                        freq_dict[prefixed],
                    )
                )
            break

len(pairs)

2633

## 6) Save all pairs and filter to MIN_FREQ

In [6]:
pairs_csv = os.path.join(OUT_DIR, "serbian_verb_pairs_hf_7001_10000.csv")
with open(pairs_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["base_verb", "prefixed_verb", "prefix", "freq_base", "freq_prefixed"])
    writer.writerows(pairs)

pairs_500 = [p for p in pairs if p[3] >= MIN_FREQ and p[4] >= MIN_FREQ]
final_csv = os.path.join(OUT_DIR, "serbian_verb_final_hf_wiki_5_7001_10000.csv")
with open(final_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["base_verb", "prefixed_verb", "prefix", "freq_base", "freq_prefixed"])
    writer.writerows(pairs_500)

pairs_csv, final_csv, len(pairs), len(pairs_500)

('data\\csv\\serbian_verb_pairs_hf_7001_10000.csv',
 'data\\csv\\serbian_verb_final_hf_wiki_5_7001_10000.csv',
 2633,
 899)